In [1]:
!pip install torch transformers graphviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 15.0 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 16.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 20.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 15.4 MB/s  0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.1
    Uninstalling click-8.4.1:
      Successfully uninstalled click-8.4.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [transformers] [transformers]ub]


In [7]:
import random
from dataclasses import dataclass, field
from typing import Any, List

random.seed(42)

@dataclass(frozen=True)
class Step:
    action: Any
    state: Any
    cost: float

class SearchProblem:

    def start_state(self) -> Any:
        raise NotImplementedError

    def successors(self, state: Any) -> List[Step]:
        raise NotImplementedError
    
    def is_end(self, state: Any) -> bool:
        raise NotImplementedError

@dataclass
class Solution:
    steps: List[Step]
    cost: float = field(init=False)

    def __post_init__(self):
        self.cost = sum(step.cost for step in self.steps)

step1 = Step(action="a", state=1, cost=1)
step2 = Step(action="b", state=2, cost=3)
solution = Solution([step1, step2])
solution.cost


4

In [9]:
# Example 1 - Travel Search Problem
# Move from 1 to N, action could be walk（+1，Cost=1）or  tram（*2，Cost=2）

class TravelSearchProblem(SearchProblem):

    def __init__(self, num_locs: int):
        self.num_locs = num_locs

    def start_state(self) -> int:
        return 1
    
    def successors(self, state: int) -> List[Step]:
        steps = []
        if state < self.num_locs:
            steps.append(Step(action="walk", state=state + 1, cost=1.0))
        if state * 2 <= self.num_locs:
            steps.append(Step(action="tram", state=state * 2, cost=2.0))
        return steps
        
    def is_end(self, state: int) -> bool:
        return state == self.num_locs

# Quick test
problem = TravelSearchProblem(num_locs=10)
print(f"Start State:{problem.start_state()}")
print(f"Successors from state 1:{problem.successors(1)}")

Start State:1
Successors from state 1:[Step(action='walk', state=2, cost=1.0), Step(action='tram', state=2, cost=2.0)]


In [11]:
# Example 2 - Limited Travel Search Problem 
# Add constrait of limited tickets to take the tram

@dataclass(frozen=True)
class TravelSearchState:
    loc: int
    tickets: int #remaining tickets


class LimitedTravelSearchProblem(SearchProblem):

    def __init__(self, num_locs: int, num_tickets: int):
        self.num_locs = num_locs
        self.num_tickets = num_tickets

    def start_state(self) -> TravelSearchState:
        return TravelSearchState(loc=1, tickets=self.num_tickets)
    
    def successors(self, state: TravelSearchState) -> List[Step]:
        steps = []
        if state.loc < self.num_locs:
            steps.append(Step(
                action="walk", 
                state=TravelSearchState(state.loc + 1, state.tickets), 
                cost=1.0))
        if state.loc * 2 <= self.num_locs and state.tickets > 0:
            steps.append(Step(
                action="tram", 
                state=TravelSearchState(state.loc * 2, state.tickets - 1), 
                cost=2.0))
        return steps
        
    def is_end(self, state: TravelSearchState) -> bool:
        return state.loc == self.num_locs

# Test limited travel problem
limited_problem = LimitedTravelSearchProblem(num_locs=10, num_tickets=1)
print(f"Limited Travel Start:{limited_problem.start_state()}")
print(f"Successors from state 1:{limited_problem.successors(TravelSearchState(1, 1))}")

Limited Travel Start:TravelSearchState(loc=1, tickets=1)
Successors from state 1:[Step(action='walk', state=TravelSearchState(loc=2, tickets=1), cost=1.0), Step(action='tram', state=TravelSearchState(loc=2, tickets=0), cost=2.0)]


In [ ]:
# Exhaustive Search

